# CUDA on an A100: A Hands-On Guide

This notebook explains what CUDA is and walks through its core features with runnable examples:

1. Querying the GPU
2. Your first kernel and the thread hierarchy
3. Vector addition with explicit memory management
4. Unified memory and grid-stride loops
5. Shared memory (tiled matrix multiplication vs. naive vs. cuBLAS)
6. Reductions with warp shuffles and atomics
7. Histograms with shared-memory atomics
8. Streams to overlap data transfers and compute
9. Tensor Cores with WMMA
10. Writing CUDA kernels from Python with CuPy
11. Asynchronous launches and how to time GPU code correctly

**Before you start:** in Colab go to *Runtime → Change runtime type* and select **A100 GPU**. The notebook also runs on other NVIDIA GPUs (T4, L4), but performance numbers will differ.

## 0. Setup and GPU check

In [1]:
!nvidia-smi

Thu Nov 12 01:55:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import subprocess, torch

assert torch.cuda.is_available(), "No GPU found. Change the runtime type to GPU (A100)."

props = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability(0)
ARCH = f"sm_{major}{minor}"   # A100 = sm_80. Used when compiling with nvcc.

print(f"GPU:                {props.name}")
print(f"Compute capability: {major}.{minor}  ->  ARCH = {ARCH}")
print(f"SMs:                {props.multi_processor_count}")
print(f"Memory:             {props.total_memory / 1e9:.1f} GB")
print(f"PyTorch:            {torch.__version__} (CUDA {torch.version.cuda})")
print(subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout.splitlines()[-1])

if "A100" not in props.name:
    print("\nNote: this is not an A100. Everything will run, but timings will differ.")

GPU:                NVIDIA A100-SXM4-80GB
Compute capability: 8.0  ->  ARCH = sm_80
SMs:                108
Memory:             85.1 GB
PyTorch:            2.14.1+cu130 (CUDA 13.0)
Build cuda_13.0.r13.0/compiler.36424714_0


## 1. What is CUDA?

**CUDA** (*Compute Unified Device Architecture*) is NVIDIA's platform for general-purpose computing on GPUs. It includes:

- **A language**: C/C++ with a few extensions (`__global__`, `<<<...>>>`, `threadIdx`, etc.).
- **A compiler**: `nvcc`, which splits CPU code (*host*) from GPU code (*device*).
- **A runtime and driver**: to allocate memory, copy data and launch work on the GPU.
- **Optimized libraries**: cuBLAS (linear algebra), cuDNN (neural networks), cuFFT, NCCL (multi-GPU communication), and more. PyTorch and TensorFlow are built on top of them.

### The core idea: host and device

- The **host** is the CPU and its RAM.
- The **device** is the GPU and its own memory (on the A100, 40 or 80 GB of HBM2e at ~1.5–2 TB/s).
- A **kernel** is a function marked `__global__` that runs on the GPU and is **executed by thousands of threads at once**, each working on different data.

Typical flow of a CUDA program:

1. Allocate GPU memory (`cudaMalloc`).
2. Copy inputs CPU → GPU (`cudaMemcpy`).
3. Launch the kernel: `my_kernel<<<blocks, threads_per_block>>>(args)`.
4. Copy results GPU → CPU.
5. Free memory (`cudaFree`).

### Execution hierarchy

```
Grid (the whole launch)
 └── Blocks (blockIdx)        -> each block runs on one SM and never migrates
      └── Warps (32 threads)   -> the real unit of execution: 32 threads run the same instruction
           └── Threads (threadIdx)
```

Each thread computes its global index as `blockIdx.x * blockDim.x + threadIdx.x`.

### Memory hierarchy (fastest to slowest)

| Memory | Scope | Approx. size on A100 | Notes |
|---|---|---|---|
| Registers | 1 thread | 64K 32-bit registers per SM | Fastest |
| Shared memory | 1 block | up to 164 KB per SM | Programmer-managed; key for data reuse |
| L2 cache | whole GPU | 40 MB | Automatic |
| Global memory (HBM) | whole GPU | 40/80 GB | Large but slow compared to the above |

### The A100 by the numbers

- 108 SMs, Ampere architecture (compute capability 8.0).
- ~19.5 TFLOPS FP32 and ~312 TFLOPS FP16/BF16 with **Tensor Cores** (units specialized for small matrix multiplies).

Most CUDA optimization boils down to **moving less data from global memory** and **keeping every SM busy**.

### How the CUDA examples run in Colab

Each example has two cells:
1. `%%writefile file.cu` saves the C++/CUDA source to a file.
2. `!nvcc -arch={ARCH} ...` compiles it for your GPU and runs it.

First, a shared header with an error-checking macro. **CUDA API calls don't throw exceptions; they return an error code you need to check.**

In [3]:
%%writefile cuda_utils.h
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

// Check the result of any CUDA API call
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err__ = (call);                                          \
        if (err__ != cudaSuccess) {                                          \
            fprintf(stderr, "CUDA error at %s:%d -> %s\n",                   \
                    __FILE__, __LINE__, cudaGetErrorString(err__));          \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

// Check for errors from a kernel launch (launches are asynchronous)
#define CUDA_CHECK_KERNEL() CUDA_CHECK(cudaGetLastError())

Overwriting cuda_utils.h


## 2. Example: querying GPU properties

The runtime API lets you ask the GPU about its capabilities. This helps you pick block sizes, decide how much shared memory to use, and so on.

In [4]:
%%writefile device_query.cu
#include "cuda_utils.h"

int main() {
    int n_devices = 0;
    CUDA_CHECK(cudaGetDeviceCount(&n_devices));

    for (int d = 0; d < n_devices; ++d) {
        cudaDeviceProp p;
        CUDA_CHECK(cudaGetDeviceProperties(&p, d));
        printf("GPU %d: %s\n", d, p.name);
        printf("  Compute capability:       %d.%d\n", p.major, p.minor);
        printf("  Multiprocessors (SMs):    %d\n", p.multiProcessorCount);
        printf("  Global memory:            %.1f GB\n", p.totalGlobalMem / 1e9);
        printf("  L2 cache:                 %d MB\n", p.l2CacheSize / (1024 * 1024));
        printf("  Shared mem per block:     %zu KB (opt-in max: %zu KB)\n",
               p.sharedMemPerBlock / 1024, p.sharedMemPerBlockOptin / 1024);
        printf("  Registers per SM:         %d\n", p.regsPerMultiprocessor);
        printf("  Max threads per block:    %d\n", p.maxThreadsPerBlock);
        printf("  Max threads per SM:       %d\n", p.maxThreadsPerMultiProcessor);
        printf("  Warp size:                %d\n", p.warpSize);
        printf("  Memory bus width:         %d bits\n", p.memoryBusWidth);
    }
    return 0;
}

Overwriting device_query.cu


In [5]:
!nvcc -arch={ARCH} -o device_query device_query.cu && ./device_query

GPU 0: NVIDIA A100-SXM4-80GB
  Compute capability:       8.0
  Multiprocessors (SMs):    108
  Global memory:            85.1 GB
  L2 cache:                 40 MB
  Shared mem per block:     48 KB (opt-in max: 163 KB)
  Registers per SM:         65536
  Max threads per block:    1024
  Max threads per SM:       2048
  Warp size:                32
  Memory bus width:         5120 bits


## 3. Example: your first kernel and the thread hierarchy

- `__global__` marks a function that runs on the GPU and is called from the CPU.
- `<<<2, 4>>>` launches **2 blocks of 4 threads** (8 threads total).
- Each thread knows where it is through `blockIdx`, `blockDim` and `threadIdx`.
- Launches are **asynchronous**: the CPU keeps going. `cudaDeviceSynchronize()` waits for the GPU to finish.

Note that **print order is not guaranteed**: blocks run in parallel and in any order.

In [6]:
%%writefile hello.cu
#include "cuda_utils.h"

__global__ void hello_gpu() {
    int global_id = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Hello from block %d, thread %d  ->  global id %d\n",
           blockIdx.x, threadIdx.x, global_id);
}

int main() {
    hello_gpu<<<2, 4>>>();          // 2 blocks x 4 threads
    CUDA_CHECK_KERNEL();
    CUDA_CHECK(cudaDeviceSynchronize());
    printf("The CPU continues after the GPU has finished.\n");
    return 0;
}

Overwriting hello.cu


In [7]:
!nvcc -arch={ARCH} -o hello hello.cu && ./hello

Hello from block 0, thread 0  ->  global id 0
Hello from block 0, thread 1  ->  global id 1
Hello from block 0, thread 2  ->  global id 2
Hello from block 0, thread 3  ->  global id 3
Hello from block 1, thread 0  ->  global id 4
Hello from block 1, thread 1  ->  global id 5
Hello from block 1, thread 2  ->  global id 6
Hello from block 1, thread 3  ->  global id 7
The CPU continues after the GPU has finished.


## 4. Example: vector addition with explicit memory management

This is the real "hello world" of CUDA. It shows the full flow: allocate, copy, launch, copy back, free.

Key points:
- **One thread per element**. Since `n` isn't always a multiple of the block size, the kernel guards with `if (i < n)`.
- Timing uses **CUDA events** (`cudaEvent_t`), which measure on the GPU itself.
- We report **effective bandwidth**: vector addition is memory-bound (2 floats read and 1 written per add), so the metric that matters is GB/s, not FLOPS.

In [8]:
%%writefile vector_add.cu
#include "cuda_utils.h"
#include <cmath>
#include <chrono>

__global__ void vector_add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int n = 1 << 26;                     // ~67 million elements
    const size_t bytes = n * sizeof(float);

    // 1. Host memory
    float *h_a = (float*)malloc(bytes), *h_b = (float*)malloc(bytes), *h_c = (float*)malloc(bytes);
    for (int i = 0; i < n; ++i) { h_a[i] = sinf(i); h_b[i] = cosf(i); }

    // CPU reference
    auto t0 = std::chrono::high_resolution_clock::now();
    for (int i = 0; i < n; ++i) h_c[i] = h_a[i] + h_b[i];
    double ms_cpu = std::chrono::duration<double, std::milli>(
        std::chrono::high_resolution_clock::now() - t0).count();

    // 2. Device memory
    float *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));

    // 3. Copy CPU -> GPU
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    // 4. Configure and launch the kernel
    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;   // round up
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n); // warm-up
    CUDA_CHECK_KERNEL();

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    const int reps = 20;
    CUDA_CHECK(cudaEventRecord(start));
    for (int r = 0; r < reps; ++r) vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    float ms_gpu = 0;
    CUDA_CHECK(cudaEventElapsedTime(&ms_gpu, start, stop));
    ms_gpu /= reps;

    // 5. Copy GPU -> CPU and verify
    float* h_out = (float*)malloc(bytes);
    CUDA_CHECK(cudaMemcpy(h_out, d_c, bytes, cudaMemcpyDeviceToHost));
    double max_err = 0;
    for (int i = 0; i < n; ++i) max_err = fmax(max_err, fabs(h_out[i] - h_c[i]));

    printf("Elements:         %d (%d blocks of %d threads)\n", n, blocks, threads);
    printf("CPU time:         %.2f ms\n", ms_cpu);
    printf("GPU kernel time:  %.3f ms  (%.0fx faster)\n", ms_gpu, ms_cpu / ms_gpu);
    printf("Bandwidth:        %.0f GB/s\n", 3.0 * bytes / (ms_gpu * 1e-3) / 1e9);
    printf("Max error:        %g\n", max_err);

    // 6. Free
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c); free(h_out);
    return 0;
}

Overwriting vector_add.cu


In [9]:
!nvcc -O3 -arch={ARCH} -o vector_add vector_add.cu && ./vector_add

Elements:         67108864 (262144 blocks of 256 threads)
CPU time:         154.06 ms
GPU kernel time:  0.480 ms  (321x faster)
Bandwidth:        1678 GB/s
Max error:        0


## 5. Example: unified memory and grid-stride loops

**Unified memory** (`cudaMallocManaged`): a single pointer that is valid on both CPU and GPU. The driver migrates pages on demand. It's convenient, but the first time the GPU touches a page there's a *page fault* and a migration, so **the first launch is slower**.

**Grid-stride loop**: instead of launching exactly one thread per element, you launch a fixed number of threads (e.g. a few blocks per SM) and each thread strides by `blockDim.x * gridDim.x`. The same kernel works for any `n`.

The kernel is **SAXPY** (`y = a*x + y`), a classic BLAS operation.

In [10]:
%%writefile saxpy_unified.cu
#include "cuda_utils.h"
#include <cmath>

__global__ void saxpy(int n, float a, const float* x, float* y) {
    // grid-stride loop: each thread handles several elements
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x)
        y[i] = a * x[i] + y[i];
}

float time_launch(int blocks, int threads, int n, float a, float* x, float* y) {
    cudaEvent_t s, e;
    cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    saxpy<<<blocks, threads>>>(n, a, x, y);
    cudaEventRecord(e);
    cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms, s, e);
    return ms;
}

int main() {
    const int n = 1 << 26;
    float *x, *y;
    CUDA_CHECK(cudaMallocManaged(&x, n * sizeof(float)));   // a single pointer!
    CUDA_CHECK(cudaMallocManaged(&y, n * sizeof(float)));

    for (int i = 0; i < n; ++i) { x[i] = 1.0f; y[i] = 2.0f; }  // CPU writes directly

    int dev, num_sms;
    CUDA_CHECK(cudaGetDevice(&dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&num_sms, cudaDevAttrMultiProcessorCount, dev));
    const int threads = 256, blocks = 32 * num_sms;   // fixed number of blocks

    float ms1 = time_launch(blocks, threads, n, 2.0f, x, y);  // with page migration
    float ms2 = time_launch(blocks, threads, n, 2.0f, x, y);  // data already on GPU
    CUDA_CHECK_KERNEL();

    // CPU reads the result directly (y = 2*1 + 2 = 4, then 2*1 + 4 = 6)
    double max_err = 0;
    for (int i = 0; i < n; ++i) max_err = fmax(max_err, fabs(y[i] - 6.0f));

    printf("Blocks: %d x %d threads for %d elements\n", blocks, threads, n);
    printf("1st launch (with page migration): %.2f ms\n", ms1);
    printf("2nd launch (data already on GPU): %.2f ms\n", ms2);
    printf("Max error: %g\n", max_err);

    cudaFree(x); cudaFree(y);
    return 0;
}

Overwriting saxpy_unified.cu


In [11]:
!nvcc -O3 -arch={ARCH} -o saxpy_unified saxpy_unified.cu && ./saxpy_unified

Blocks: 3456 x 256 threads for 67108864 elements
1st launch (with page migration): 80.76 ms
2nd launch (data already on GPU): 0.53 ms
Max error: 0


## 6. Example: shared memory with matrix multiplication

This is **the** classic CUDA optimization example. We compare three versions of `C = A × B` (2048×2048 FP32 matrices):

1. **Naive**: each thread computes one element of C by reading a row of A and a column of B from global memory. Every value is read from global memory many times.
2. **Tiled with shared memory**: each 32×32 block loads one tile of A and one tile of B into shared memory **once**, and every thread in the block reuses them. `__syncthreads()` is a barrier: no thread moves on until all have arrived, so nobody reads a half-loaded tile.
3. **cuBLAS**: NVIDIA's library, with years of hand-tuning. Used as the reference.

Note: cuBLAS is *column-major* (Fortran-style). To multiply *row-major* matrices we use the identity `Cᵀ = Bᵀ Aᵀ`, which amounts to passing B and A in swapped order.

In [12]:
%%writefile matmul.cu
#include "cuda_utils.h"
#include <cublas_v2.h>
#include <cmath>
#include <vector>

#define TILE 32

// 1. Naive version
__global__ void matmul_naive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float acc = 0.f;
        for (int k = 0; k < N; ++k) acc += A[row * N + k] * B[k * N + col];
        C[row * N + col] = acc;
    }
}

// 2. Shared-memory tiled version
__global__ void matmul_tiled(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float acc = 0.f;

    for (int t = 0; t < (N + TILE - 1) / TILE; ++t) {
        int a_col = t * TILE + threadIdx.x;
        int b_row = t * TILE + threadIdx.y;
        // Each thread loads ONE element of each tile
        As[threadIdx.y][threadIdx.x] = (row < N && a_col < N) ? A[row * N + a_col] : 0.f;
        Bs[threadIdx.y][threadIdx.x] = (b_row < N && col < N) ? B[b_row * N + col] : 0.f;
        __syncthreads();                      // wait until the tile is fully loaded
        for (int k = 0; k < TILE; ++k) acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();                      // wait before overwriting the tile
    }
    if (row < N && col < N) C[row * N + col] = acc;
}

int main() {
    const int N = 2048;
    const size_t bytes = (size_t)N * N * sizeof(float);
    const double flops = 2.0 * N * N * N;
    const int reps = 10;

    std::vector<float> hA(N * N), hB(N * N), hC1(N * N), hC2(N * N), hC3(N * N);
    srand(0);
    for (auto& v : hA) v = rand() / (float)RAND_MAX - 0.5f;
    for (auto& v : hB) v = rand() / (float)RAND_MAX - 0.5f;

    float *dA, *dB, *dC;
    CUDA_CHECK(cudaMalloc(&dA, bytes)); CUDA_CHECK(cudaMalloc(&dB, bytes)); CUDA_CHECK(cudaMalloc(&dC, bytes));
    CUDA_CHECK(cudaMemcpy(dA, hA.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB.data(), bytes, cudaMemcpyHostToDevice));

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    float ms;
    dim3 threads(TILE, TILE);
    dim3 blocks((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);

    // --- Naive ---
    matmul_naive<<<blocks, threads>>>(dA, dB, dC, N);
    cudaEventRecord(s);
    for (int r = 0; r < reps; ++r) matmul_naive<<<blocks, threads>>>(dA, dB, dC, N);
    cudaEventRecord(e); cudaEventSynchronize(e); cudaEventElapsedTime(&ms, s, e);
    CUDA_CHECK_KERNEL();
    printf("Naive:         %7.2f ms  %6.0f GFLOPS\n", ms / reps, flops / (ms / reps * 1e-3) / 1e9);
    CUDA_CHECK(cudaMemcpy(hC1.data(), dC, bytes, cudaMemcpyDeviceToHost));

    // --- Shared memory ---
    matmul_tiled<<<blocks, threads>>>(dA, dB, dC, N);
    cudaEventRecord(s);
    for (int r = 0; r < reps; ++r) matmul_tiled<<<blocks, threads>>>(dA, dB, dC, N);
    cudaEventRecord(e); cudaEventSynchronize(e); cudaEventElapsedTime(&ms, s, e);
    CUDA_CHECK_KERNEL();
    printf("Shared memory: %7.2f ms  %6.0f GFLOPS\n", ms / reps, flops / (ms / reps * 1e-3) / 1e9);
    CUDA_CHECK(cudaMemcpy(hC2.data(), dC, bytes, cudaMemcpyDeviceToHost));

    // --- cuBLAS (pure FP32) ---
    cublasHandle_t h; cublasCreate(&h);
    const float alpha = 1.f, beta = 0.f;
    // Row-major: C = A*B  <=>  column-major: C^T = B^T * A^T
    cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, N, N, N, &alpha, dB, N, dA, N, &beta, dC, N);
    cudaEventRecord(s);
    for (int r = 0; r < reps; ++r)
        cublasSgemm(h, CUBLAS_OP_N, CUBLAS_OP_N, N, N, N, &alpha, dB, N, dA, N, &beta, dC, N);
    cudaEventRecord(e); cudaEventSynchronize(e); cudaEventElapsedTime(&ms, s, e);
    printf("cuBLAS:        %7.2f ms  %6.0f GFLOPS\n", ms / reps, flops / (ms / reps * 1e-3) / 1e9);
    CUDA_CHECK(cudaMemcpy(hC3.data(), dC, bytes, cudaMemcpyDeviceToHost));

    double err1 = 0, err2 = 0;
    for (int i = 0; i < N * N; ++i) {
        err1 = fmax(err1, fabs(hC1[i] - hC3[i]));
        err2 = fmax(err2, fabs(hC2[i] - hC3[i]));
    }
    printf("\nMax difference vs cuBLAS: naive %.2e, shared %.2e\n", err1, err2);

    cublasDestroy(h); cudaFree(dA); cudaFree(dB); cudaFree(dC);
    return 0;
}

Overwriting matmul.cu


In [13]:
!nvcc -O3 -arch={ARCH} -o matmul matmul.cu -lcublas && ./matmul

Naive:            5.68 ms    3027 GFLOPS
Shared memory:    3.89 ms    4411 GFLOPS
cuBLAS:           1.19 ms   14451 GFLOPS

Max difference vs cuBLAS: naive 0.00e+00, shared 0.00e+00


**What to look for:** the shared-memory version should be noticeably faster than the naive one, and cuBLAS faster still. cuBLAS adds more techniques: each thread computes several outputs (*register tiling*), vectorized loads, asynchronous copies and *double buffering*.

## 7. Example: reduction with warp shuffles and atomics

Summing all elements of an array (a **reduction**) is tricky in parallel because every thread has to combine its result with the others. This kernel works at three levels:

1. **Per thread**: each thread sums several elements with a grid-stride loop.
2. **Per warp**: `__shfl_down_sync` lets the 32 threads of a warp exchange values **directly between registers**, without touching memory. After 5 steps (16, 8, 4, 2, 1) lane 0 holds the warp's sum.
3. **Per block**: each warp writes its result to shared memory, the first warp adds those up, and an `atomicAdd` accumulates the block result into global memory.

`atomicAdd` guarantees the read-add-write is indivisible, even when many blocks write to the same address at once.

In [14]:
%%writefile reduce.cu
#include "cuda_utils.h"
#include <vector>

__inline__ __device__ float warp_sum(float v) {
    // Sum across the 32 threads of a warp using shuffles (no memory)
    for (int offset = 16; offset > 0; offset >>= 1)
        v += __shfl_down_sync(0xffffffff, v, offset);
    return v;
}

__global__ void reduce_sum(const float* x, float* out, int n) {
    // 1. Per-thread sum
    float s = 0.f;
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x)
        s += x[i];

    // 2. Per-warp sum
    s = warp_sum(s);

    // 3. Per-block sum
    __shared__ float warp_sums[32];
    int lane = threadIdx.x % 32, warp_id = threadIdx.x / 32;
    if (lane == 0) warp_sums[warp_id] = s;
    __syncthreads();
    if (warp_id == 0) {
        s = (threadIdx.x < blockDim.x / 32) ? warp_sums[lane] : 0.f;
        s = warp_sum(s);
        if (lane == 0) atomicAdd(out, s);   // 4. Global atomic accumulation
    }
}

int main() {
    const int n = 1 << 26;
    std::vector<float> h(n);
    double ref = 0;
    for (int i = 0; i < n; ++i) { h[i] = (i % 7) * 0.25f; ref += h[i]; }

    float *dx, *dout;
    CUDA_CHECK(cudaMalloc(&dx, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&dout, sizeof(float)));
    CUDA_CHECK(cudaMemcpy(dx, h.data(), n * sizeof(float), cudaMemcpyHostToDevice));

    int dev, sms; cudaGetDevice(&dev);
    cudaDeviceGetAttribute(&sms, cudaDevAttrMultiProcessorCount, dev);
    const int threads = 512, blocks = 4 * sms;

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    CUDA_CHECK(cudaMemset(dout, 0, sizeof(float)));
    cudaEventRecord(s);
    reduce_sum<<<blocks, threads>>>(dx, dout, n);
    cudaEventRecord(e); cudaEventSynchronize(e);
    CUDA_CHECK_KERNEL();
    float ms; cudaEventElapsedTime(&ms, s, e);

    float result;
    CUDA_CHECK(cudaMemcpy(&result, dout, sizeof(float), cudaMemcpyDeviceToHost));
    printf("GPU sum:  %.1f\n", result);
    printf("CPU sum:  %.1f\n", ref);
    printf("Relative error: %.2e  (FP32 accumulates rounding error)\n", fabs(result - ref) / ref);
    printf("Time: %.3f ms  ->  %.0f GB/s\n", ms, n * 4.0 / (ms * 1e-3) / 1e9);
    cudaFree(dx); cudaFree(dout);
    return 0;
}

Overwriting reduce.cu


In [15]:
!nvcc -O3 -arch={ARCH} -o reduce reduce.cu && ./reduce

GPU sum:  50331652.0
CPU sum:  50331646.5
Relative error: 1.09e-07  (FP32 accumulates rounding error)
Time: 0.403 ms  ->  665 GB/s


## 8. Example: histogram with shared-memory atomics

A histogram is another case where many threads write to the same locations. We compare two strategies:

- **Global atomics**: every thread does `atomicAdd` on the same 256 counters in global memory. Lots of **contention** (threads waiting to write to the same place).
- **Shared-memory atomics**: each block builds a private histogram in shared memory (much faster) and adds it to the global one at the end. Global contention drops from "one atomic per element" to "256 atomics per block".

In [16]:
%%writefile histogram.cu
#include "cuda_utils.h"
#include <vector>

#define BINS 256

__global__ void hist_global(const unsigned char* data, int n, unsigned int* hist) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x)
        atomicAdd(&hist[data[i]], 1u);
}

__global__ void hist_shared(const unsigned char* data, int n, unsigned int* hist) {
    __shared__ unsigned int local[BINS];
    for (int i = threadIdx.x; i < BINS; i += blockDim.x) local[i] = 0;
    __syncthreads();
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x)
        atomicAdd(&local[data[i]], 1u);            // shared-memory atomic
    __syncthreads();
    for (int i = threadIdx.x; i < BINS; i += blockDim.x)
        atomicAdd(&hist[i], local[i]);             // only 256 global atomics per block
}

int main() {
    const int n = 1 << 27;
    std::vector<unsigned char> h(n);
    std::vector<unsigned int> ref(BINS, 0), out(BINS);
    srand(1);
    // Skewed data (many small values): increases contention
    for (int i = 0; i < n; ++i) { h[i] = (rand() % 256) * (rand() % 256) / 256; ref[h[i]]++; }

    unsigned char* d; unsigned int* dh;
    CUDA_CHECK(cudaMalloc(&d, n));
    CUDA_CHECK(cudaMalloc(&dh, BINS * sizeof(unsigned int)));
    CUDA_CHECK(cudaMemcpy(d, h.data(), n, cudaMemcpyHostToDevice));

    int dev, sms; cudaGetDevice(&dev);
    cudaDeviceGetAttribute(&sms, cudaDevAttrMultiProcessorCount, dev);
    const int threads = 256, blocks = 8 * sms;
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    float ms;

    const char* names[2] = {"Global atomics", "Shared atomics"};
    for (int v = 0; v < 2; ++v) {
        CUDA_CHECK(cudaMemset(dh, 0, BINS * sizeof(unsigned int)));
        cudaEventRecord(s);
        if (v == 0) hist_global<<<blocks, threads>>>(d, n, dh);
        else        hist_shared<<<blocks, threads>>>(d, n, dh);
        cudaEventRecord(e); cudaEventSynchronize(e);
        CUDA_CHECK_KERNEL();
        cudaEventElapsedTime(&ms, s, e);
        CUDA_CHECK(cudaMemcpy(out.data(), dh, BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));
        bool ok = (out == ref);
        printf("%-16s %7.2f ms   correct: %s\n", names[v], ms, ok ? "yes" : "NO");
    }
    printf("First bins: %u %u %u %u ...\n", out[0], out[1], out[2], out[3]);
    cudaFree(d); cudaFree(dh);
    return 0;
}

Overwriting histogram.cu


In [17]:
!nvcc -O3 -arch={ARCH} -o histogram histogram.cu && ./histogram

Global atomics     51.80 ms   correct: yes
Shared atomics      0.40 ms   correct: yes
First bins: 4030699 2660711 2403108 2250679 ...


## 9. Example: streams to overlap copies and compute

A **stream** is a queue of operations the GPU runs in order. Operations in **different streams** can run concurrently. The A100 has copy engines separate from its SMs, so it can **copy data while computing**.

Strategy: split the work into 4 chunks. While chunk 1 is being processed, chunk 2 is being copied, and so on (*pipelining*).

Requirements:
- Host memory must be **pinned** (`cudaMallocHost`, memory the OS can't page out). With regular memory, `cudaMemcpyAsync` isn't truly asynchronous.
- Use `cudaMemcpyAsync` and pass the stream as the 4th parameter of `<<<...>>>`.

In [18]:
%%writefile streams.cu
#include "cuda_utils.h"
#include <cmath>

__global__ void work(float* x, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float v = x[i];
        for (int k = 0; k < 60; ++k) v = sinf(v) * 0.5f + cosf(v) * 0.5f;  // artificial compute
        x[i] = v;
    }
}

int main() {
    const int n = 1 << 26, S = 4, chunk = n / S;
    const size_t bytes = n * sizeof(float), cbytes = chunk * sizeof(float);

    float *h, *d;
    CUDA_CHECK(cudaMallocHost(&h, bytes));      // pinned memory
    CUDA_CHECK(cudaMalloc(&d, bytes));
    for (int i = 0; i < n; ++i) h[i] = i * 1e-7f;

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    float ms_seq, ms_str;

    // --- Sequential: copy everything, compute everything, copy everything ---
    cudaEventRecord(s);
    CUDA_CHECK(cudaMemcpy(d, h, bytes, cudaMemcpyHostToDevice));
    work<<<(n + 255) / 256, 256>>>(d, n);
    CUDA_CHECK(cudaMemcpy(h, d, bytes, cudaMemcpyDeviceToHost));
    cudaEventRecord(e); cudaEventSynchronize(e); cudaEventElapsedTime(&ms_seq, s, e);

    // --- With streams: each chunk in its own stream ---
    cudaStream_t st[S];
    for (int i = 0; i < S; ++i) CUDA_CHECK(cudaStreamCreate(&st[i]));
    cudaEventRecord(s);
    for (int i = 0; i < S; ++i) {
        int off = i * chunk;
        CUDA_CHECK(cudaMemcpyAsync(d + off, h + off, cbytes, cudaMemcpyHostToDevice, st[i]));
        work<<<(chunk + 255) / 256, 256, 0, st[i]>>>(d + off, chunk);
        CUDA_CHECK(cudaMemcpyAsync(h + off, d + off, cbytes, cudaMemcpyDeviceToHost, st[i]));
    }
    cudaEventRecord(e);              // the default stream waits for the others
    cudaEventSynchronize(e);
    CUDA_CHECK_KERNEL();
    cudaEventElapsedTime(&ms_str, s, e);

    printf("Sequential:        %.2f ms\n", ms_seq);
    printf("With %d streams:    %.2f ms  (%.2fx)\n", S, ms_str, ms_seq / ms_str);

    for (int i = 0; i < S; ++i) cudaStreamDestroy(st[i]);
    cudaFreeHost(h); cudaFree(d);
    return 0;
}

Overwriting streams.cu


In [19]:
!nvcc -O3 -arch={ARCH} -o streams streams.cu && ./streams

Sequential:        81.33 ms
With 4 streams:    56.17 ms  (1.45x)


## 10. Example: Tensor Cores with WMMA

**Tensor Cores** are GPU units that multiply small matrices in a single instruction. On the A100 they're the difference between ~19.5 TFLOPS (regular FP32) and ~312 TFLOPS (FP16/BF16). They are what makes neural network training fast.

The **WMMA** API (*Warp Matrix Multiply-Accumulate*) exposes them in CUDA C++:
- A **whole warp** (32 threads) cooperates to multiply 16×16×16 tiles.
- Data is loaded into **fragments** (`wmma::fragment`), whose layout across threads is opaque.
- FP16 (`half`) inputs, FP32 accumulation.

This kernel is simple (it reads straight from global memory, no shared memory), but it already uses Tensor Cores. Production libraries (cuBLAS, CUTLASS) combine this with everything shown above.

In [20]:
%%writefile wmma.cu
#include "cuda_utils.h"
#include <cuda_fp16.h>
#include <mma.h>
#include <vector>
#include <cmath>
using namespace nvcuda;

// Each warp computes one 16x16 tile of C
__global__ void wmma_gemm(const half* A, const half* B, float* C, int M, int N, int K) {
    int warp_m = (blockIdx.x * blockDim.x + threadIdx.x) / 32;
    int warp_n = blockIdx.y * blockDim.y + threadIdx.y;
    int row = warp_m * 16, col = warp_n * 16;
    if (row >= M || col >= N) return;   // the whole warp exits together

    wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::row_major> a_frag;
    wmma::fragment<wmma::matrix_b, 16, 16, 16, half, wmma::row_major> b_frag;
    wmma::fragment<wmma::accumulator, 16, 16, 16, float> c_frag;
    wmma::fill_fragment(c_frag, 0.0f);

    for (int k = 0; k < K; k += 16) {
        wmma::load_matrix_sync(a_frag, A + row * K + k, K);
        wmma::load_matrix_sync(b_frag, B + k * N + col, N);
        wmma::mma_sync(c_frag, a_frag, b_frag, c_frag);   // Tensor Cores!
    }
    wmma::store_matrix_sync(C + row * N + col, c_frag, N, wmma::mem_row_major);
}

int main() {
    const int M = 4096, N = 4096, K = 4096;
    std::vector<half> hA(M * K), hB(K * N);
    std::vector<float> hC(M * N);
    srand(0);
    for (auto& v : hA) v = __float2half(rand() / (float)RAND_MAX - 0.5f);
    for (auto& v : hB) v = __float2half(rand() / (float)RAND_MAX - 0.5f);

    half *dA, *dB; float* dC;
    CUDA_CHECK(cudaMalloc(&dA, M * K * sizeof(half)));
    CUDA_CHECK(cudaMalloc(&dB, K * N * sizeof(half)));
    CUDA_CHECK(cudaMalloc(&dC, M * N * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(dA, hA.data(), M * K * sizeof(half), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(dB, hB.data(), K * N * sizeof(half), cudaMemcpyHostToDevice));

    dim3 threads(128, 4);                                 // 4x4 = 16 warps per block
    dim3 blocks((M + 16 * 4 - 1) / (16 * 4), (N + 16 * 4 - 1) / (16 * 4));

    wmma_gemm<<<blocks, threads>>>(dA, dB, dC, M, N, K);   // warm-up
    CUDA_CHECK_KERNEL();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    const int reps = 10;
    cudaEventRecord(s);
    for (int r = 0; r < reps; ++r) wmma_gemm<<<blocks, threads>>>(dA, dB, dC, M, N, K);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms, s, e); ms /= reps;
    CUDA_CHECK(cudaMemcpy(hC.data(), dC, M * N * sizeof(float), cudaMemcpyDeviceToHost));

    // Check a few random entries against the CPU
    double max_rel = 0;
    for (int t = 0; t < 20; ++t) {
        int i = rand() % M, j = rand() % N;
        double ref = 0;
        for (int k = 0; k < K; ++k) ref += (double)__half2float(hA[i * K + k]) * __half2float(hB[k * N + j]);
        max_rel = fmax(max_rel, fabs(hC[i * N + j] - ref) / (fabs(ref) + 1e-3));
    }
    printf("WMMA FP16 (%dx%dx%d): %.2f ms  ->  %.1f TFLOPS\n", M, N, K, ms, 2.0 * M * N * K / (ms * 1e-3) / 1e12);
    printf("Max relative error (20 samples): %.2e\n", max_rel);
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    return 0;
}

Overwriting wmma.cu


In [21]:
!nvcc -O3 -arch={ARCH} -o wmma wmma.cu && ./wmma

WMMA FP16 (4096x4096x4096): 9.46 ms  ->  14.5 TFLOPS
Max relative error (20 samples): 3.23e-05


Compare it with PyTorch, which calls cuBLAS with Tensor Cores and every optimization under the hood:

In [22]:
M = N = K = 4096
a = torch.randn(M, K, device="cuda", dtype=torch.float16)
b = torch.randn(K, N, device="cuda", dtype=torch.float16)
for _ in range(3): a @ b
torch.cuda.synchronize()

start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
start.record()
for _ in range(20): a @ b
end.record(); torch.cuda.synchronize()
ms = start.elapsed_time(end) / 20
print(f"PyTorch/cuBLAS FP16: {ms:.2f} ms  ->  {2 * M * N * K / (ms * 1e-3) / 1e12:.1f} TFLOPS")

PyTorch/cuBLAS FP16: 0.52 ms  ->  265.4 TFLOPS


## 11. Example: CUDA from Python with CuPy

You don't have to leave Python to write CUDA kernels. **CuPy** compiles CUDA C++ at runtime (via NVRTC) and launches it on GPU arrays. And with **DLPack**, PyTorch tensors can be shared with CuPy without copying.

In [23]:
try:
    import cupy as cp
except ImportError:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    import cupy as cp

leaky_relu_kernel = cp.RawKernel(r'''
extern "C" __global__
void leaky_relu(const float* x, float* y, float alpha, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float v = x[i];
        y[i] = v > 0.f ? v : alpha * v;
    }
}
''', 'leaky_relu')

# A PyTorch tensor...
x_torch = torch.randn(10_000_000, device="cuda")
# ...shared with CuPy without copying memory (DLPack)
x_cp = cp.from_dlpack(x_torch)
y_cp = cp.empty_like(x_cp)

n = x_cp.size
threads = 256
blocks = (n + threads - 1) // threads
leaky_relu_kernel((blocks,), (threads,), (x_cp, y_cp, cp.float32(0.1), cp.int32(n)))

# Back to PyTorch, also without copying
y_torch = torch.from_dlpack(y_cp)
ref = torch.nn.functional.leaky_relu(x_torch, 0.1)
print("Matches PyTorch?", torch.allclose(y_torch, ref))
print("Same memory buffer as CuPy:", y_torch.data_ptr() == y_cp.data.ptr)

Matches PyTorch? True
Same memory buffer as CuPy: True


## 12. An important detail: kernel launches are asynchronous

When PyTorch (or your CUDA code) launches a kernel, the CPU **doesn't wait** for it to finish. If you time with `time.time()` without synchronizing, you only measure how long it took to *enqueue* the work. This is one of the most common mistakes when benchmarking GPU code.

In [24]:
import time

x = torch.randn(8192, 8192, device="cuda")
torch.cuda.synchronize()

t0 = time.perf_counter()
y = x @ x
t1 = time.perf_counter()
print(f"Without sync: {(t1 - t0) * 1e3:7.3f} ms  <- just the time to enqueue")

torch.cuda.synchronize()
t0 = time.perf_counter()
y = x @ x
torch.cuda.synchronize()
t1 = time.perf_counter()
print(f"With sync:    {(t1 - t0) * 1e3:7.3f} ms  <- the real time")

Without sync:  61.012 ms  <- just the time to enqueue
With sync:     58.682 ms  <- the real time


---
## Summary

| Concept | What you saw |
|---|---|
| Kernels and thread hierarchy | `__global__`, `<<<blocks, threads>>>`, `blockIdx`/`threadIdx` |
| Memory management | `cudaMalloc`, `cudaMemcpy`, unified memory, pinned memory |
| Shared memory | Tiling for data reuse, `__syncthreads()` |
| Warps | `__shfl_down_sync` for register-level communication |
| Atomics | `atomicAdd` in global and shared memory, contention |
| Concurrency | Streams to overlap copies and compute |
| Tensor Cores | WMMA API, FP16 inputs with FP32 accumulation |
| Python interop | CuPy `RawKernel`, DLPack zero-copy sharing with PyTorch |
| Benchmarking | CUDA events, asynchronous launches, synchronizing before timing |

### Further reading

- *CUDA C++ Programming Guide* (official NVIDIA documentation).
- *Programming Massively Parallel Processors* (Hwu, Kirk, El Hajj), the standard textbook.
- CUTLASS and CuTe (NVIDIA's open-source templates for high-performance GEMM and attention kernels).
- Nsight Compute and Nsight Systems for profiling kernels.